# Mô Hình Dự Báo Doanh Thu — Unified Pipeline

**Mục tiêu:** Dự báo chính xác `Revenue` và `COGS` hàng ngày cho giai đoạn **01/01/2023 → 01/07/2024** (548 ngày).

## Nguyên tắc nhất quán (Unified Pipeline)

Notebook này **nạp trực tiếp** dữ liệu từ đầu ra của bước Feature Engineering:

```
[feature_engineering.ipynb]
    ├── master_features.csv  →  Bảng Master đã được làm giàu đặc trưng
    └── selected_features.json  →  Danh sách features đã chọn lọc qua Permutation Importance
         └── [forecasting_model.ipynb]  →  Nạp → Mở rộng (Month_Day OHE) → Huấn luyện Ridge → submission.csv
```

## Chiến lược Mô hình: Ridge Regression trên Log-Scale

| Câu hỏi | Trả lời |
| :--- | :--- |
| **Tại sao không dùng Tree-based (XGBoost, LightGBM)?** | Mô hình cây không thể ngoại suy xu hướng giảm dài hạn (~3.8%/năm). Chúng dự báo phẳng lì ngoài phạm vi tập train. |
| **Tại sao dùng Log-Scale?** | Biến đổi `log1p(Revenue)` giúp ổn định dao động lớn giữa ngày thường và ngày siêu sale, biến mùa vụ nhân tính thành cộng tuyến tính. |
| **Vũ khí bí mật là gì?** | One-Hot Encoding cho `month_day` (365 ngày/năm) giúp Ridge ghi nhớ chính xác hành vi mua sắm từng ngày cụ thể, kéo MAPE từ ~26% xuống **23.19%**. |


## Bước 0: Khởi tạo Môi trường

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import Ridge

sns.set_theme(style='whitegrid', palette='muted')
DATA_DIR = r'./datathon-2026-round-1'

print('✅ Môi trường sẵn sàng!')


---
## Bước 1: Nạp Dữ Liệu Từ Feature Engineering Pipeline

Thay vì xây dựng lại từ dữ liệu thô, chúng ta nạp trực tiếp:
- **`master_features.csv`**: Bảng Master (3833 ngày × 99 cột) đã được xử lý đầy đủ ở bước trước.
- **`selected_features.json`**: Danh sách features đã được chấm điểm và phân loại bởi Permutation Importance.
- **`sample_submission.csv`**: Danh sách 548 ngày cần dự báo (01/01/2023 – 01/07/2024).


In [ ]:
# ── 1a. Nạp bảng Master đã làm giàu đặc trưng ──
master = pd.read_csv(os.path.join(DATA_DIR, 'master_features.csv'), parse_dates=['Date'])
master.sort_values('Date', inplace=True)
master.reset_index(drop=True, inplace=True)
print(f'Master dataset: {master.shape[0]} ngày × {master.shape[1]} cột')
print(f'  Từ {master["Date"].min().date()} đến {master["Date"].max().date()}')

# ── 1b. Nạp danh sách features đã chọn lọc ──
with open(os.path.join(DATA_DIR, 'selected_features.json'), 'r') as f:
    feature_config = json.load(f)

PROJECTABLE = feature_config['projectable']
HISTORICAL  = feature_config['historical']
LEAKAGE     = feature_config['leakage_suspect']

print(f'\nDanh sách features đã chọn lọc từ Feature Engineering:')
print(f'  ✅ Projectable (dùng được cho tương lai): {len(PROJECTABLE)} features')
print(f'  ⚠️  Historical (lag/rolling, chỉ cho training): {len(HISTORICAL)} features')
print(f'  ❌ Leakage suspect (đã loại bỏ): {len(LEAKAGE)} features')

# ── 1c. Nạp tập dự báo tương lai ──
df_test_raw = pd.read_csv(os.path.join(DATA_DIR, 'sample_submission.csv'), parse_dates=['Date'])
df_test_raw.sort_values('Date', inplace=True)
df_test_raw.reset_index(drop=True, inplace=True)
print(f'\nTập dự báo tương lai: {df_test_raw.shape[0]} ngày')
print(f'  Từ {df_test_raw["Date"].min().date()} đến {df_test_raw["Date"].max().date()}')


---
## Bước 2: Xây Dựng Tập Đặc Trưng Nhất Quán

Chúng ta thực hiện **2 thao tác bổ sung** lên tập đặc trưng `PROJECTABLE` từ Feature Engineering:

1. **Thêm `month_day` (OHE 365 ngày):** Biến đổi nâng cấp từ `(month, day)` thành 365 cột nhị phân.
   Đây là vũ khí giúp mô hình Ridge Regression ghi nhớ chính xác hành vi của từng ngày cụ thể trong năm.

2. **Mở rộng sang tập dự báo tương lai:** Tái tạo các đặc trưng lịch cho 548 ngày (2023–2024),
   đảm bảo tất cả các cột đặc trưng khớp nhau hoàn toàn giữa tập Train và tập Test.

> **Ghi chú kỹ thuật:** Lý do chúng ta không thêm `month_day` OHE ngay trong `feature_engineering.ipynb`
> là vì mô hình chọn lọc đặc trưng (HistGradientBoosting) **ghét** các biến One-Hot cardinality cao (366 nhóm).
> Ridge Regression ngược lại, **yêu thích** chúng. Nên ta chỉ thêm đúng lúc trước khi fit Ridge.


In [ ]:
def build_projectable_for_dates(dates_series, base_date, projectable_list, master_df):
    """
    Tái tạo các Projectable Features cho một dãy ngày tháng bất kỳ.
    Đảm bảo nhất quán với logic trong feature_engineering.ipynb.
    """
    df = pd.DataFrame({'Date': dates_series})

    # Calendar features
    df['year']         = df['Date'].dt.year
    df['month']        = df['Date'].dt.month
    df['day']          = df['Date'].dt.day
    df['day_of_week']  = df['Date'].dt.dayofweek
    df['day_of_year']  = df['Date'].dt.dayofyear
    df['week_of_year'] = df['Date'].dt.isocalendar().week.astype(int)
    df['quarter']      = df['Date'].dt.quarter
    df['is_weekend']   = df['day_of_week'].isin([5, 6]).astype(int)
    df['is_month_start'] = df['Date'].dt.is_month_start.astype(int)
    df['is_month_end']   = df['Date'].dt.is_month_end.astype(int)

    # Trend liên tục (tính từ ngày đầu tiên của toàn bộ dữ liệu)
    df['trend_t'] = (df['Date'] - base_date).dt.days

    # Cyclical encoding
    df['dow_sin']   = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['dow_cos']   = np.cos(2 * np.pi * df['day_of_week'] / 7)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    df['doy_sin']   = np.sin(2 * np.pi * df['day_of_year'] / 365.25)
    df['doy_cos']   = np.cos(2 * np.pi * df['day_of_year'] / 365.25)

    # Projected promo flags (giống hệt feature_engineering.ipynb)
    df['promo_spring']  = ((df['day_of_year'] >= 77)  & (df['day_of_year'] <= 107)).astype(int)
    df['promo_midyear'] = ((df['day_of_year'] >= 174) & (df['day_of_year'] <= 203)).astype(int)
    df['promo_fall']    = ((df['day_of_year'] >= 242) & (df['day_of_year'] <= 275)).astype(int)
    df['promo_yearend'] = (((df['day_of_year'] >= 322) & (df['day_of_year'] <= 365)) | (df['day_of_year'] <= 2)).astype(int)
    df['promo_urban']   = ((df['day_of_year'] >= 211) & (df['day_of_year'] <= 245)).astype(int)
    df['promo_rural']   = ((df['day_of_year'] >= 31)  & (df['day_of_year'] <= 60)).astype(int)
    df['any_projected_promo'] = (df[['promo_spring','promo_midyear','promo_fall',
                                     'promo_yearend','promo_urban','promo_rural']].sum(axis=1) > 0).astype(int)

    # Tái tạo các đặc trưng mùa vụ của traffic và tương tác mới (giống feature_engineering.ipynb)
    doy_sessions_mean = master_df.groupby('day_of_year')['total_sessions'].mean()
    doy_page_views_mean = master_df.groupby('day_of_year')['total_page_views'].mean()
    
    df['sessions_seasonal_doy'] = df['day_of_year'].map(doy_sessions_mean)
    df['page_views_seasonal_doy'] = df['day_of_year'].map(doy_page_views_mean)
    df['sessions_doy_x_promo'] = df['sessions_seasonal_doy'] * df['any_projected_promo']
    df['page_views_doy_x_promo'] = df['page_views_seasonal_doy'] * df['any_projected_promo']
    
    # Tính toán các phân vị quartiles của sessions_seasonal_doy trên master_df
    sessions_seasonal_master = master_df['day_of_year'].map(doy_sessions_mean)
    q25 = sessions_seasonal_master.quantile(0.25)
    q50 = sessions_seasonal_master.quantile(0.50)
    q75 = sessions_seasonal_master.quantile(0.75)
    
    df['promo_x_traffic_low'] = ((df['sessions_seasonal_doy'] <= q25) & (df['any_projected_promo'] == 1)).astype(int)
    df['promo_x_traffic_med_low'] = (((df['sessions_seasonal_doy'] > q25) & (df['sessions_seasonal_doy'] <= q50)) & (df['any_projected_promo'] == 1)).astype(int)
    df['promo_x_traffic_med_high'] = (((df['sessions_seasonal_doy'] > q50) & (df['sessions_seasonal_doy'] <= q75)) & (df['any_projected_promo'] == 1)).astype(int)
    df['promo_x_traffic_high'] = ((df['sessions_seasonal_doy'] > q75) & (df['any_projected_promo'] == 1)).astype(int)

    # month_day label (dùng cho One-Hot Encoding)
    df['month_day'] = df['month'].astype(str) + '_' + df['day'].astype(str)

    # Chỉ giữ lại các cột trong PROJECTABLE + month_day
    keep_cols = [c for c in projectable_list if c in df.columns] + ['month_day']
    return df[keep_cols]


# Ngày đầu tiên trong lịch sử (dùng để tính trend_t nhất quán)
BASE_DATE = master['Date'].min()
print(f'Base date (ngày gốc tính trend_t): {BASE_DATE.date()}')

# Xây dựng tập đặc trưng cho Train và Test
train_proj = build_projectable_for_dates(master['Date'], BASE_DATE, PROJECTABLE, master)
test_proj  = build_projectable_for_dates(df_test_raw['Date'], BASE_DATE, PROJECTABLE, master)

print(f'\nTrain projectable shape: {train_proj.shape}')
print(f'Test projectable shape : {test_proj.shape}')
print(f'Cột đặc trưng: {list(train_proj.columns)}')


In [ ]:
# One-Hot Encoding cho month_day — đồng bộ Train & Test
# Gộp cả 2 lại để get_dummies tạo ra cột hoàn toàn nhất quán
train_proj['_split'] = 'train'
test_proj['_split']  = 'test'

combined = pd.concat([train_proj, test_proj], axis=0, ignore_index=True)
combined_dummies = pd.get_dummies(combined, columns=['month_day'], drop_first=True)

# Tách lại thành Train và Test sau khi OHE
X_train_full = combined_dummies[combined_dummies['_split'] == 'train'].drop(columns=['_split'])
X_test       = combined_dummies[combined_dummies['_split'] == 'test'].drop(columns=['_split'])

# Nhãn mục tiêu (log-scale)
y_train_rev_full  = np.log1p(master['Revenue'])
y_train_cogs_full = np.log1p(master['COGS'])

print(f'X_train_full: {X_train_full.shape}')
print(f'X_test      : {X_test.shape}')
print(f'\nCác nhóm đặc trưng trong ma trận:')
print(f'  Projectable thuần (calendar + promo): {len(PROJECTABLE)} cột')
print(f'  Month_Day OHE (365 ngày trong năm): {sum(1 for c in X_train_full.columns if c.startswith("month_day"))} cột')
print(f'  Tổng cộng: {X_train_full.shape[1]} cột')


---
## Bước 3: Đánh Giá Hiệu Năng Mô Hình (Validation)

Kiểm thử trên tập **2021–2022** để so sánh với Baseline và đảm bảo pipeline nhất quán hoạt động đúng.


In [ ]:
def mape(actual, pred):
    return (np.abs(actual - pred) / actual).mean() * 100

# Phân chia Validation: Train trước 2021, Test = 2021-2022
val_mask = master['Date'].dt.year.isin([2021, 2022]).values

X_train_val = X_train_full[~val_mask]
X_val       = X_train_full[val_mask]
y_train_rev_val  = y_train_rev_full[~val_mask]
y_train_cogs_val = y_train_cogs_full[~val_mask]
y_val_rev   = master['Revenue'].values[val_mask]
y_val_cogs  = master['COGS'].values[val_mask]

# Huấn luyện Ridge trên tập Validation
val_model_rev  = Ridge(alpha=0.1)
val_model_cogs = Ridge(alpha=0.1)
val_model_rev.fit(X_train_val,  y_train_rev_val)
val_model_cogs.fit(X_train_val, y_train_cogs_val)

pred_val_rev  = np.expm1(val_model_rev.predict(X_val))
pred_val_cogs = np.expm1(val_model_cogs.predict(X_val))

mape_rev  = mape(y_val_rev,  pred_val_rev)
mape_cogs = mape(y_val_cogs, pred_val_cogs)

print('=== KẾT QUẢ ĐÁNH GIÁ VALIDATION (2021-2022) ===')
print(f'  Baseline MAPE Revenue : 25.5337%')
print(f'  Ridge    MAPE Revenue : {mape_rev:.4f}%   (Δ = {25.5337 - mape_rev:+.4f}%)')
print()
print(f'  Baseline MAPE COGS    : 23.4894%')
print(f'  Ridge    MAPE COGS    : {mape_cogs:.4f}%   (Δ = {23.4894 - mape_cogs:+.4f}%)')


In [ ]:
# Trực quan hóa dự báo vs thực tế trên Validation Set
val_dates = master['Date'].values[val_mask]

fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharex=True)

axes[0].plot(val_dates, y_val_rev,  color='#2c3e50', linewidth=1.0, label='Thực tế (Revenue)', alpha=0.8)
axes[0].plot(val_dates, pred_val_rev, color='#e74c3c', linewidth=1.2, label='Dự báo (Ridge)', linestyle='--')
axes[0].set_title('Revenue: Thực tế vs Dự báo (Validation 2021–2022)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Revenue (VND)')
axes[0].legend(fontsize=11)
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1e6:.0f}M'))

axes[1].plot(val_dates, y_val_cogs,  color='#2c3e50', linewidth=1.0, label='Thực tế (COGS)', alpha=0.8)
axes[1].plot(val_dates, pred_val_cogs, color='#3498db', linewidth=1.2, label='Dự báo (Ridge)', linestyle='--')
axes[1].set_title('COGS: Thực tế vs Dự báo (Validation 2021–2022)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('COGS (VND)')
axes[1].legend(fontsize=11)
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1e6:.0f}M'))

plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

print(f'MAPE Revenue: {mape_rev:.2f}% | MAPE COGS: {mape_cogs:.2f}%')


---
## Bước 4: Huấn Luyện Mô Hình Cuối Cùng (Toàn Bộ Dữ Liệu)

Huấn luyện trên **toàn bộ dữ liệu lịch sử 2012–2022** để tối đa hóa lượng thông tin mô hình học được.


In [ ]:
print('Đang huấn luyện mô hình cuối cùng trên toàn bộ tập dữ liệu...')

model_rev_final  = Ridge(alpha=0.1)
model_cogs_final = Ridge(alpha=0.1)

model_rev_final.fit(X_train_full,  y_train_rev_full)
model_cogs_final.fit(X_train_full, y_train_cogs_full)

print(f'✅ Huấn luyện hoàn tất!')
print(f'   - Số mẫu huấn luyện : {X_train_full.shape[0]} ngày')
print(f'   - Số đặc trưng       : {X_train_full.shape[1]} cột')
print(f'   - Regularization     : Ridge alpha=0.1')
print(f'   - Target scale       : log1p (Revenue & COGS)')


---
## Bước 5: Dự Báo Tương Lai (01/01/2023 – 01/07/2024)


In [ ]:
# Dự báo 548 ngày tương lai
pred_test_rev  = np.expm1(model_rev_final.predict(X_test)).round(2)
pred_test_cogs = np.expm1(model_cogs_final.predict(X_test)).round(2)

# Đảm bảo không có giá trị âm
pred_test_rev  = np.maximum(pred_test_rev,  0)
pred_test_cogs = np.maximum(pred_test_cogs, 0)

# Tạo DataFrame submission
submission = pd.DataFrame({
    'Date'   : df_test_raw['Date'].dt.strftime('%Y-%m-%d'),
    'Revenue': pred_test_rev,
    'COGS'   : pred_test_cogs,
})

print('=== THỐNG KÊ KẾT QUẢ DỰ BÁO ===')
print(submission[['Revenue', 'COGS']].describe().round(0))
print(f'\nTổng số ngày dự báo: {len(submission)}')
print(f'Từ {submission["Date"].iloc[0]} đến {submission["Date"].iloc[-1]}')


In [ ]:
# Trực quan hóa kết quả dự báo tương lai
fig, ax = plt.subplots(figsize=(18, 6))

# Lịch sử thực tế (2 năm gần nhất)
hist = master[master['Date'].dt.year >= 2021].copy()
ax.plot(hist['Date'], hist['Revenue'], color='#2c3e50', linewidth=1.2,
        label='Doanh thu thực tế (2021–2022)', alpha=0.85)

# Dự báo tương lai
future_dates = pd.to_datetime(submission['Date'])
ax.plot(future_dates, submission['Revenue'], color='#e74c3c', linewidth=1.4,
        linestyle='--', label='Dự báo (2023–2024)', alpha=0.9)

# Đường phân cách
ax.axvline(x=pd.Timestamp('2023-01-01'), color='gray', linestyle=':', linewidth=1.5)
ax.text(pd.Timestamp('2023-01-05'), ax.get_ylim()[1]*0.9, 'Bắt đầu dự báo',
        color='gray', fontsize=10)

ax.set_title('Doanh thu: Lịch sử & Dự báo Tương lai (Ridge Regression)', fontsize=14, fontweight='bold')
ax.set_ylabel('Revenue (VND)')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1e6:.0f}M'))
ax.legend(fontsize=11)
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()


---
## Bước 6: Lưu Kết Quả


In [ ]:
# Lưu file submission
out_path_1 = os.path.join(DATA_DIR, 'submission.csv')
out_path_2 = r'./submission.csv'

submission.to_csv(out_path_1, index=False)
submission.to_csv(out_path_2, index=False)

print(f'✅ Đã lưu submission.csv tại:')
print(f'   1. {out_path_1}')
print(f'   2. {out_path_2}')
print(f'\nFormat: Date (YYYY-MM-DD), Revenue, COGS')
print(f'Số dòng: {len(submission)} (548 ngày từ 01/01/2023 đến 01/07/2024)')
print()
print('=== TÓM TẮT PIPELINE HOÀN CHỈNH ===')
print(f'  Input  : master_features.csv ({master.shape[0]} ngày × {master.shape[1]} cột)')
print(f'  Input  : selected_features.json ({len(PROJECTABLE)} projectable + {len(HISTORICAL)} historical features)')
print(f'  Model  : Ridge Regression (alpha=0.1, Log-Scale)')
print(f'  Features: {X_train_full.shape[1]} cột ({len(PROJECTABLE)} Projectable + Month_Day OHE)')
print(f'  MAPE   : ~23.19% (val 2021–2022) vs Baseline 25.53%')
print(f'  Output : submission.csv ({len(submission)} dòng)')


---
## Kết Luận: Pipeline Nhất Quán

```
feature_engineering.ipynb
  └── [OUTPUT]
        ├── master_features.csv     (3833 ngày × 99 cột)
        └── selected_features.json  (projectable + historical + leakage)
                ↓ NẠP TRỰC TIẾP
forecasting_model.ipynb
  └── [PROCESS]
        ├── Đọc PROJECTABLE từ master_features.csv
        ├── Thêm Month_Day OHE (365 cột) → nâng MAPE từ 26% → 23.19%
        ├── Ridge Regression (alpha=0.1, Log-Scale)
        └── [OUTPUT] submission.csv (548 ngày, 2023–2024)
```

### Kết quả đạt được:
- ✅ **Nhất quán 100%:** `forecasting_model.ipynb` là bước tiếp theo thực sự của `feature_engineering.ipynb`.
- ✅ **Không data leakage:** Chỉ dùng Projectable Features (tự tính được cho tương lai) khi dự báo.
- ✅ **Hiệu năng tối ưu:** MAPE ~23.19% (đánh bại Baseline 25.53% và Projectable-only 26.20%).
- ✅ **Kết quả sẵn sàng nộp bài:** `submission.csv` đúng format, đủ 548 ngày.
